# 🚗 Dynamic Pricing Engine for Urban Parking Lots
**Capstone Project – Summer Analytics 2025**  
👤 **Author:** Ankit Kumar  
🎓 **Domain:** Urban Mobility & Operations Research | CSE B.Tech Capstone  

---

## 📌 Executive Summary
Urban parking congestion leads to significant economic loss, excess carbon emissions, and driver frustration. This project introduces a comprehensive real-time **Dynamic Pricing & Rerouting Engine** designed for modern urban smart cities. 

### Core Innovations:
1. **Model 1 – Baseline Linear Pricing:** Simple occupancy-driven dynamic adjustment.
2. **Model 2 – Multi-Factor Demand-Based Dynamic Pricing:** Evaluates composite demand using occupancy ratio, queue length, surrounding traffic density, special event indicators, and vehicle classification.
3. **Model 3 – Geo-Distance Competitive Pricing:** Incorporates real-time nearby competitor parking rates using Haversine distance calculations.
4. **Smart Rerouting Engine:** Recommends optimal neighboring parking lots when a target lot reaches saturation capacity.
5. **Interactive Visualization:** Leverages Bokeh & Plotly for interactive simulation.

In [ ]:
# Step 1: Environment Setup & Package Imports
import math
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool

warnings.filterwarnings('ignore')
output_notebook()

---
## 📊 Step 2: Dataset Loading & Exploratory Data Analysis (EDA)
We load the synthetic urban parking dataset comprising 14 parking hubs across the city.

In [ ]:
# Load Dataset
from src.data_generator import generate_parking_dataset

df = generate_parking_dataset(num_records=500, seed=42)
df.to_csv('dataset.csv', index=False)
print(f"Loaded dataset with {len(df)} records and {df.shape[1]} features.")
df.head()

In [ ]:
# Exploratory Data Analysis (EDA)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['Occupancy'] / df['Capacity'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Occupancy Ratio Distribution')
axes[0].set_xlabel('Occupancy / Capacity')

sns.countplot(data=df, x='TrafficConditionNearby', ax=axes[1], palette='Set2')
axes[1].set_title('Nearby Traffic Condition Counts')

sns.boxplot(data=df, x='VehicleType', y='BasePrice', ax=axes[2], palette='Pastel1')
axes[2].set_title('Base Price by Vehicle Type')

plt.tight_layout()
plt.show()

---
## 🧮 Step 3: Mathematical Formulations & Pricing Models Implementation

### 1. Model 1: Baseline Linear Pricing
\[
P_{t+1} = P_t + \alpha \cdot \left(\frac{\text{Occupancy}}{\text{Capacity}}\right)
\]

In [ ]:
from src.pricing_engine import linear_pricing, calculate_demand_score, demand_based_pricing, competitive_pricing

# Apply Model 1
df['Model1_Price'] = df.apply(lambda r: linear_pricing(r['BasePrice'], r['Occupancy'], r['Capacity']), axis=1)
df[['LotName', 'Occupancy', 'Capacity', 'BasePrice', 'Model1_Price']].head()

### 2. Model 2: Multi-Factor Demand-Based Dynamic Pricing
\[
D = \alpha \cdot \left(\frac{\text{Occupancy}}{\text{Capacity}}\right) + \beta \cdot \text{Queue} - \gamma \cdot \text{Traffic} + \delta \cdot \text{Special} + \epsilon \cdot \text{VehicleWeight}
\]
\[
P_t = P_{\text{base}} \cdot \left(1 + \lambda \cdot (\hat{D} - 0.5)\right)
\]

In [ ]:
# Apply Model 2 & Model 3
df['DemandScore'] = df.apply(lambda r: calculate_demand_score(
    r['Occupancy'], r['Capacity'], r['QueueLength'], r['TrafficConditionNearby'], r['IsSpecialDay'], r['VehicleType']
), axis=1)

df['Model2_Price'] = df.apply(lambda r: demand_based_pricing(r['BasePrice'], r['DemandScore']), axis=1)
df['Model3_Price'] = df.apply(lambda r: competitive_pricing(r['Model2_Price'], r['CompetitorPrice']), axis=1)

df[['LotName', 'DemandScore', 'BasePrice', 'Model1_Price', 'Model2_Price', 'Model3_Price']].head()

---
## 📈 Step 4: Interactive Bokeh Visualization & Time Simulation

In [ ]:
# Interactive Bokeh Simulation
sample_lot = df[df['LotID'] == 'LOT_01'].head(10).copy().reset_index(drop=True)
sample_lot['TimeStep'] = [f"T{i+1}" for i in range(len(sample_lot))]

source = ColumnDataSource(sample_lot)

p = figure(title="Dynamic Price Trajectory over Time (LOT_01)", x_range=list(sample_lot['TimeStep']),
           x_axis_label='Time Step', y_axis_label='Price ($)', width=750, height=400)

p.line(x='TimeStep', y='Model1_Price', source=source, legend_label='Model 1 (Linear)', line_width=2, color='gray', line_dash='dashed')
p.line(x='TimeStep', y='Model2_Price', source=source, legend_label='Model 2 (Demand)', line_width=3, color='navy')
p.line(x='TimeStep', y='Model3_Price', source=source, legend_label='Model 3 (Competitive)', line_width=2, color='green')
p.circle(x='TimeStep', y='Model2_Price', source=source, size=8, color='red')

hover = HoverTool()
hover.tooltips = [("Time", "@TimeStep"), ("Demand Score", "@DemandScore{0.00}"), ("Model 2 Price", "$@Model2_Price{0.00}")]
p.add_tools(hover)

p.legend.location = "top_left"
show(p)

---
## 🧭 Step 5: Smart Lot Rerouting Test

In [ ]:
from src.rerouting import find_alternative_lots

all_lots_status = df.groupby('LotID').first().reset_index()
lots_list = all_lots_status.to_dict('records')

# Simulate Lot 1 as full
lots_list[0]['Occupancy'] = lots_list[0]['Capacity']

alternatives = find_alternative_lots(lots_list[0]['LotID'], lots_list, max_distance_km=10.0)
print(f"Target Lot: {lots_list[0]['LotName']} (FULL: {lots_list[0]['Occupancy']}/{lots_list[0]['Capacity']})")
print("Top Alternative Recommendations:")
pd.DataFrame(alternatives)

---
## ✅ Conclusion & Future Scope
- Successfully developed production-grade 3-tier pricing models for urban parking.
- Integrated spatial Haversine distance rerouting engine for traffic congestion reduction.
- Future work includes integrating Pathway streaming engine for sub-second IoT sensor pipelines.